# 🚀 Result — RNV1 高速強化学習スタジオ (Google Colab Pro A100)

**Harbor Range (50 × 200 m)** の完全同一3D物理環境を **A100 GPU 上で数千アリーナ同時並列化** し、
ランダム初期化（一から）の状態から **CTDE MAPPO + Recurrent GRU + PFSP 自己対戦リーグ** で次世代 NPC AI **`RNV1`** を超高速学習・強化します。

---

### ▶ 使い方（スタートを押すだけ！）
1. 上部メニュー **「ランタイム」→「ランタイムのタイプを変更」** で **ハードウェア アクセラレータ: `A100 GPU`** が選択されていることを確認（ノートブック側で自動指定済み）。
2. 上部メニュー **「ランタイム」→「すべて実行」(Ctrl+F9 / ⌘+F9)** を押すだけ！
3. 学習完了後、実機ゲームエンジン (`dist/js/match.js`) で `NV2.6 Force` 等と自動ベンチマークし、**`rnv1-weights.json` が自動ダウンロード** されます。
4. ゲーム画面の **対戦設定 → 実験場 → `重み読込 (.json)`** で読み込めば即座に最強 `RNV1` と対戦できます！

In [ ]:
#@title ⚙️ 1. スタジオ設定 & A100環境自動セットアップ（そのまま実行OK）
PRESET = "a100_pro_fast" #@param ["a100_quick_test", "a100_pro_fast", "a100_pro_max"]
TEAM_SIZE = 3 #@param [1, 3, 5] {type:"raw"}
REPO_URL = "https://github.com/sc8z35a-collab/Result.git" #@param {type:"string"}
REPO_BRANCH = "auto" #@param ["auto", "main", "feat/rnv1-colab-a100-studio"]
SAVE_TO_GOOGLE_DRIVE = False #@param {type:"boolean"}
RUN_JS_ARENA_BENCHMARK = True #@param {type:"boolean"}
AUTO_DOWNLOAD_WEIGHTS = True #@param {type:"boolean"}
FRESH_START = True #@param {type:"boolean"}

import os, subprocess, sys, shutil
from pathlib import Path
import torch

def run_cmd(cmd, **kwargs):
    res = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if res.returncode != 0:
        print(f"  ❌ Command failed ({res.returncode}): {' '.join(map(str, cmd))}")
        if res.stdout:
            print("  [stdout]", res.stdout[-1500:])
        if res.stderr:
            print("  [stderr]", res.stderr[-1500:])
        raise subprocess.CalledProcessError(res.returncode, cmd, res.stdout, res.stderr)
    return res

print("=" * 72)
print("  RNV1 Studio — Google Colab Pro A100 Pre-Flight Check")
print("=" * 72)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"  GPU Detected : {gpu_name} ({vram_gb:.1f} GB VRAM) | BF16={bf16_ok}")
    if "A100" not in gpu_name and "H100" not in gpu_name:
        print("  ⚠️  注意: A100以外のGPUが割り当てられています。ランタイムを A100 に切り替えると最高速で動作します。")
else:
    print("  ⚠️  GPUが未接続です！ 上部メニュー「ランタイム → ランタイムのタイプを変更 → A100 GPU」を選択してください。")

# Optional Google Drive mirror (default False so Run All is 100% hands-free)
DRIVE_DIR = ""
if SAVE_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_DIR = "/content/drive/MyDrive/Result_RNV1"
        Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
        print(f"  Google Drive : {DRIVE_DIR}")
    except Exception as e:
        print(f"  Google Drive mount skipped: {e}")

# Clone or update Result repository automatically (handles shallow single-branch fetch cleanly!)
REPO_DIR = Path("/content/Result")
if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    run_cmd(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    run_cmd(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])

def branch_has_studio(branch: str) -> bool:
    r = subprocess.run(
        ["git", "-C", str(REPO_DIR), "ls-remote", "--heads", "origin", f"refs/heads/{branch}"],
        capture_output=True, text=True
    )
    if r.returncode != 0 or not r.stdout.strip():
        return False
    # Explicit refspec + FETCH_HEAD avoids shallow single-branch 'origin/<branch>' missing-ref error!
    run_cmd(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", f"+refs/heads/{branch}:refs/remotes/origin/{branch}"])
    run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", branch, "FETCH_HEAD"])
    run_cmd(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"])
    return (REPO_DIR / "training" / "rnv1_studio" / "trainer.py").exists()

if REPO_BRANCH == "auto":
    if not branch_has_studio("main"):
        assert branch_has_studio("feat/rnv1-colab-a100-studio"), "RNV1 studio files not found on main or feature branch!"
        print("  Branch       : feat/rnv1-colab-a100-studio (auto-selected)")
    else:
        print("  Branch       : main (auto-selected)")
else:
    assert branch_has_studio(REPO_BRANCH), f"Branch {REPO_BRANCH} missing training/rnv1_studio!"
    print(f"  Branch       : {REPO_BRANCH}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Verify Node.js >= 18 availability for exact parity & real-engine JS arena benchmark
def get_node_major():
    try:
        r = subprocess.run(["node", "-v"], capture_output=True, text=True)
        if r.returncode == 0:
            v = r.stdout.strip().lstrip("v")
            return int(v.split(".")[0]), r.stdout.strip()
    except Exception:
        pass
    return 0, "none"

node_major, node_str = get_node_major()
if node_major >= 18:
    print(f"  Node.js      : {node_str}")
else:
    print(f"  Node.js      : {node_str} -> installing Node.js 20 LTS for real-engine JS benchmark...")
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y -qq nodejs", shell=True, check=False)
    node_major, node_str = get_node_major()
    print(f"  Node.js      : {node_str}")

# Run exact map & neural parity verification before training starts
if node_major >= 18:
    print("  Running Python <-> JS exact map & neural parity check...")
    p_res = run_cmd([sys.executable, "-m", "training.rnv1_studio.test_parity"])
    for ln in p_res.stdout.strip().splitlines():
        print("   ", ln)
else:
    print("  ⚠️ Node.js 未検出のため JS パリティ検証をスキップして学習へ進みます。")

print("=" * 72)
print("  ✅ セットアップ完了！そのまま次のセルで一から高速学習が始まります。")

In [ ]:
#@title 🔥 2. RNV1 一からA100超高速学習スタート（CTDE MAPPO + Recurrent GRU + PFSPリーグ）
import os, sys, json
from pathlib import Path
from IPython.display import Image, display

os.chdir("/content/Result")
if "/content/Result" not in sys.path:
    sys.path.insert(0, "/content/Result")

from training.rnv1_studio.trainer import PRESETS, StudioConfig, train_rnv1

OUT_DIR = "/content/rnv1_outputs"
cfg = StudioConfig(preset=PRESET)
for k, v in PRESETS[PRESET].items():
    setattr(cfg, k, v)
cfg.team_size = int(TEAM_SIZE)
cfg.output_dir = OUT_DIR
cfg.drive_dir = DRIVE_DIR
cfg.fresh_start = bool(FRESH_START)

# Launch full from-scratch A100 GPU-vectorized RL training!
result = train_rnv1(cfg)

dash_png = Path(OUT_DIR) / "training_dashboard.png"
if dash_png.exists():
    display(Image(filename=str(dash_png)))

In [ ]:
#@title 🏆 3. 実機ゲームエンジン (dist/js/match.js) 360Hz 直接対戦ベンチマーク
import os, subprocess
from pathlib import Path

os.chdir("/content/Result")
OUT_DIR = Path("/content/rnv1_outputs")
best_w = OUT_DIR / "rnv1-weights-best.json"
final_w = OUT_DIR / "rnv1-weights.json"
eval_w = best_w if best_w.exists() else final_w

if RUN_JS_ARENA_BENCHMARK and eval_w.exists():
    print(f"Evaluating {eval_w.name} inside real dist/js/match.js against all Result AI generations...")
    subprocess.run([
        "node", "training/rnv1_studio/eval_js_arena.mjs",
        "--weights", str(eval_w),
        "--team-size", str(TEAM_SIZE),
        "--matches", "4",
        "--seconds", "50",
        "--gun", "MS10",
        "--out", str(OUT_DIR / "js_arena_benchmark.json"),
    ], check=True)
else:
    print("JS Arena Benchmark skipped or weights file not found.")

In [ ]:
#@title 📦 4. 学習済み RNV1 重み (rnv1-weights.json) 自動ダウンロード & 導入ガイド
import json, shutil
from pathlib import Path

OUT_DIR = Path("/content/rnv1_outputs")
best_w = OUT_DIR / "rnv1-weights-best.json"
final_w = OUT_DIR / "rnv1-weights.json"
release_w = OUT_DIR / "rnv1-weights-release.json"

chosen = best_w if best_w.exists() else final_w
assert chosen.exists(), "学習済み重みファイルが見つかりません。セル2が正常完了しているか確認してください。"

shutil.copy2(chosen, release_w)
# Also keep canonical filename `rnv1-weights.json` pointing to best weights
if chosen != final_w:
    shutil.copy2(chosen, final_w)

meta = json.loads(final_w.read_text(encoding="utf-8")).get("meta", {})
size_kb = final_w.stat().st_size / 1024.0

print("=" * 72)
print("  🎉 RNV1 学習済み重みの書き出し完了！")
print("=" * 72)
print(f"  File       : {final_w} ({size_kb:.1f} KB · Int8量子化 · ゼロ依存)")
print(f"  Version    : RNV1 | Steps: {meta.get('steps', 0):,} | Origin: {meta.get('origin', 'A100')}")
if meta.get("evalWinRate") is not None:
    print(f"  Eval Score : WinRate={meta['evalWinRate']*100:.1f}% | K/D={meta.get('evalKD', 0):.2f}")
print("-" * 72)
print("  【ゲームへの反映方法（2通り）】")
print("   方法A（最速1秒）:")
print("     Result を開く → 対戦設定 →「実験場」→「重み読込 (.json)」から")
print("     ダウンロードした rnv1-weights.json を選択！（自動で通常戦RNV1もONになります）")
print("   方法B（リポジトリへ永久反映）:")
print("     ダウンロードした rnv1-weights.json を dist/assets/rnv1-weights.json に配置して Push")
print("=" * 72)

if AUTO_DOWNLOAD_WEIGHTS:
    try:
        from google.colab import files
        files.download(str(final_w))
    except Exception as e:
        print(f"自動ダウンロードをスキップしました ({e})。左のファイル一覧から {final_w} をダウンロードできます。")